# Pricing from JSON

End-to-end book MC pricing via `pricing_engine`.

**JSON fields & products:** [`data/pricing_json_schema.yaml`](../../data/pricing_json_schema.yaml)



| File | Role |
|------|------|
| `data/market_snapshot*.json` | one or more market files (looped in order); same book for all |
| `data/options_book.json` | option specs (`options` list); generate with [`generate_options_book.ipynb`](generate_options_book.ipynb) |

## How to use this notebook

1. Build the pybind module (above).
2. Set parameters in the first code cell (`MAX_HORIZON_YEARS`, `MC_SAMPLES`, …).
3. Optionally set `GENERATE_TEST_SNAPSHOTS = N` to write N shifted copies of the base market (asof +1 business day each).
4. Loop: for every `data/market_snapshot*.json` → calibrate, simulate (horizon from that `asof`), `price_all`.
5. Inspect the summary cell: per-file errors and timing.



## Parameters (first code cell)

| Variable | Role |
|------|------|
| `MARKET_DATA_DIR`, `OPTIONS_JSON` | Input paths |
| `GENERATE_TEST_SNAPSHOTS` | `0` = use existing files only; `N` = write `market_snapshot_shift_00.json` … from base (`asof` +1 business day per step) |
| `MAX_HORIZON_YEARS` | Cap on MC horizon per market: `min(max book expiry, asof + N calendar years)` |
| `MC_SAMPLES` | Paths per simulation |
| `DYNAMICS` | `"lsv"` (default) or `"lv"` — Bergomi params come from the market JSON (`bergomi_k/nu/rho`) |
| `MC_THREADS` | OpenMP workers for MC simulate + `price_all` (written to `OMP_NUM_THREADS` before import; C++ default 6 if unset) |



## Constraints

- Dates: ISO `YYYY-MM-DD`; calendar is TARGET (use business days). All option dates must be strictly after `market["asof"]`. Alternatively set `expiry_years` (e.g. `4.0`): resolved as `asof + round(365·T)` days, adjusted to the first TARGET business day (Following). Mutually exclusive with `expiry`.
- MC grid: `expiry` and every `observation_dates` entry must fall on a business day included in the simulated path (daily TARGET schedule to the sim horizon). Weekends/holidays → `not on save path`.
- Monitoring: path-dependent products default to monthly observations when `observation_dates` is omitted; set `observation_frequency: "daily"` to monitor on every simulated business day (mutually exclusive with `observation_dates`).
- Horizon: simulation ends at `min(latest book date, asof + MAX_HORIZON_YEARS)` (also considers `observation_dates`). Options beyond that horizon cannot be priced.
- `id`: must be unique in the book (the notebook keys results by `id`).
- `product`: one of `european`, `digital`, `digital_accrual`, `asian`, `barrier`, `lookback`, `autocall`, see YAML for per-product fields.
- Order: call `simulate_paths` before `price_all`; both LV and LSV only need `calibration()`.
- Strikes / barriers: prefer `*_fraction_of_spot`; absolute `strike` / `barrier_*` also supported.

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

# OpenMP workers for simulate_paths + price_all (must be set before import pricing_engine)
MC_THREADS = 6
os.environ["OMP_NUM_THREADS"] = str(MC_THREADS)


REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent.parent
elif not (REPO / "CMakeLists.txt").exists():
    for p in REPO.parents:
        if (p / "CMakeLists.txt").exists():
            REPO = p
            break

sys.path.insert(0, str(REPO / "build-std"))

import pricing_engine as pe

MARKET_DATA_DIR = REPO / "data"
MARKET_JSON_BASE = MARKET_DATA_DIR / "market_snapshot.json"
OPTIONS_JSON = REPO / "data" / "options_book.json"
MAX_HORIZON_YEARS = 5.0
MC_SAMPLES = 2000000
DYNAMICS = "lsv"
SEED = 41
RUN_VALIDATION = True

GENERATE_TEST_SNAPSHOTS = 0

In [2]:
def parse_iso(value: str) -> date:
    y, m, d = value.split("-")
    return date(int(y), int(m), int(d))


def format_iso(d: date) -> str:
    return f"{d.year:04d}-{d.month:02d}-{d.day:02d}"


def write_shifted_market_snapshots(base_path: Path, out_dir: Path, count: int) -> list[Path]:
    import QuantLib as ql

    base = json.loads(base_path.read_text())
    y, m, d = map(int, base["asof"].split("-"))
    cal = ql.TARGET()
    asof_ql = ql.Date(d, m, y)
    written: list[Path] = []
    for i in range(count):
        snap = dict(base)
        snap["asof"] = format_iso(date(asof_ql.year(), int(asof_ql.month()), asof_ql.dayOfMonth()))
        path = out_dir / f"market_snapshot_shift_{i:02d}.json"
        path.write_text(json.dumps(snap, indent=2) + "\n")
        written.append(path)
        if i + 1 < count:
            asof_ql = cal.advance(asof_ql, 1, ql.Days, ql.Following)
    return written


def simulation_horizon(specs: list[dict], ctx, max_years: float) -> str:
    latest = max(parse_iso(ctx.resolve_expiry_years(s["expiry_years"])) for s in specs)
    for s in specs:
        for obs in s.get("observation_dates") or []:
            if obs:
                latest = max(latest, parse_iso(obs))
    ay, am, ad = map(int, ctx.today.split("-"))
    cap = date(ay + int(max_years), am, min(ad, 28))
    return format_iso(min(latest, cap))


def count_ok(results) -> int:
    return len([r for r in results if r.status == "ok"])


if GENERATE_TEST_SNAPSHOTS:
    paths = write_shifted_market_snapshots(
        MARKET_JSON_BASE, MARKET_DATA_DIR, GENERATE_TEST_SNAPSHOTS
    )
    print(f"wrote {len(paths)} test snapshots:")
    for p in paths:
        print(f"  {p.name}")

specs = json.loads(OPTIONS_JSON.read_text())["options"]
market_files = sorted(MARKET_DATA_DIR.glob("market_snapshot*.json"))
if not market_files:
    raise FileNotFoundError(f"no market_snapshot*.json in {MARKET_DATA_DIR}")

print(f"options: {len(specs)}")
print(f"market files ({len(market_files)}):")
for p in market_files:
    asof = json.loads(p.read_text())["asof"]
    print(f"  {p.name}  asof={asof}")

options: 249
market files (1):
  market_snapshot.json  asof=2025-12-01


In [3]:
run_summaries = []

for market_path in market_files:
    market = json.loads(market_path.read_text())
    print(f"\n=== {market_path.name}  asof={market['asof']} ===")

    ctx = pe.PricingContext.from_tables(**market)
    ctx.preprocessing()
    ctx.calibration(run_validation=RUN_VALIDATION)

    sim_expiry = simulation_horizon(specs, ctx, MAX_HORIZON_YEARS)
    t0 = time.perf_counter()
    sim = ctx.simulate_paths(sim_expiry, mc_samples=MC_SAMPLES, dynamics=DYNAMICS, seed=SEED)
    sim_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    results = ctx.price_all(specs)
    price_ms = (time.perf_counter() - t0) * 1000

    n_ok = count_ok(results)
    n_err = len(results) - n_ok
    print(f"horizon={sim_expiry}  sim={sim_ms:.0f} ms  price_all={price_ms:.0f} ms  ok={n_ok}  err={n_err}")

    if n_err:
        for r in results:
            if r.status != "ok":
                print(f"  ERR {r.id}: {r.status[:120]}")
                break

    run_summaries.append({
        "market_file": market_path.name,
        "asof": market["asof"],
        "horizon": sim_expiry,
        "sim_ms": sim_ms,
        "price_ms": price_ms,
        "priced": n_ok,
        "errors": n_err,
        "results": results,
    })


=== market_snapshot.json  asof=2025-12-01 ===
static arbitrage: PASS
fit implied vol: PASS
horizon=2027-12-01  sim=8575 ms  price_all=13445 ms  ok=249  err=0


In [4]:
print("\n--- summary ---")
for s in run_summaries:
    print(
        f"{s['market_file']:32s}  asof={s['asof']}  horizon={s['horizon']}  "
        f"ok={s['priced']}  err={s['errors']}  sim={s['sim_ms']:.0f}ms  price={s['price_ms']:.0f}ms"
    )

# last market: quick price dump (optional)
last = run_summaries[-1]
priced_ok = [r for r in last["results"] if r.status == "ok"]
print(f"\nlast run ({last['market_file']}): {len(priced_ok)} prices")
for r in priced_ok:
    print(f"  {r.id}: {r.value:.4f}  (stderr={r.stderr:.4f})")



--- summary ---
market_snapshot.json              asof=2025-12-01  horizon=2027-12-01  ok=249  err=0  sim=8575ms  price=13445ms

last run (market_snapshot.json): 249 prices
  digital_cash_0.80_0.5y: 0.9421  (stderr=0.0002)
  digital_asset_0.80_0.5y: 7854.5332  (stderr=1.3548)
  digital_cash_0.85_0.5y: 0.9122  (stderr=0.0002)
  digital_asset_0.85_0.5y: 7654.4591  (stderr=1.6501)
  digital_cash_0.90_0.5y: 0.8591  (stderr=0.0002)
  digital_asset_0.90_0.5y: 7276.8618  (stderr=2.0526)
  digital_cash_0.95_0.5y: 0.7642  (stderr=0.0003)
  digital_asset_0.95_0.5y: 6564.7715  (stderr=2.5482)
  digital_cash_1.00_0.5y: 0.6073  (stderr=0.0003)
  digital_asset_1.00_0.5y: 5323.5980  (stderr=3.0015)
  digital_cash_1.05_0.5y: 0.3963  (stderr=0.0003)
  digital_asset_1.05_0.5y: 3571.7276  (stderr=3.0976)
  digital_cash_1.10_0.5y: 0.2019  (stderr=0.0003)
  digital_asset_1.10_0.5y: 1881.0160  (stderr=2.6313)
  digital_cash_1.15_0.5y: 0.0810  (stderr=0.0002)
  digital_asset_1.15_0.5y: 782.6837  (stderr=1.8